# Internal Fruit Morphology Analysis in Cranberry

In this tutorial, we demonstrate how to analyze the internal morphology of fruits using `FruitInternalAnalyzer`, a tool for extracting morphology and color measurements from cross-sectional images of cranberry slices.

The first step is to import `FruitInternalAnalyzer` from Traitly and create the `cranberry` object, which will contain everything needed for the analysis. The object can be named however you prefer.

We then initialize the class by specifying the image location through the `path` parameter.


In [ ]:
from traitly.fruit_phenotyping import FruitInternalAnalyzer

In [ ]:
image_path = "./cranberry_slices.jpg"
cranberry = FruitInternalAnalyzer(path = image_path)

First, we load the image into the object using `load_image()`. By default, the image will be displayed on screen (`plot=True`). Once loaded, it can be accessed through `cranberry.img`. For more details on the data stored in the object, refer to the [class attributes] section.

In [ ]:
cranberry.load_image() 

Next, we run `setup_measurements()` to define the diameter of the size references (black circles) and, optionally, read the QR code on the label. Setting `plot_reference=True` allows us to inspect the reference detection and the pixel diameter of each circle in detail.

As shown in the output, a strip of circles (`Ref 1`) composed of 6 circles was detected. Before computing the average, `setup_measurements()` removes any circles whose standard deviation exceeds 2, to avoid noise from poorly detected or atypical circles. In this case, 5 out of 6 circles were used, yielding a mean diameter of 218 px. This value is divided by the actual mean diameter in centimeters to obtain the pixel-per-cm density, which will be used to convert pixels to centimeters in subsequent analyses.

In [ ]:
cranberry.setup_measurements(detect_label = True,
                            diameter_cm = 1.7, 
                            plot_reference = True)

We then generate a binary mask of the fruits and locules using `generate_fruit_mask()`, where locules appear in black and the rest of the fruit in white.

In [ ]:
cranberry.generate_fruit_mask()

After that, we detect the fruits in the mask with `detect_fruits()` and do a quick visual inspection using `plot=True`. The image displays each fruit's outline in green, the locule contours in pink, and the internal pericarp area in cyan. This visualization allows you to verify the detection and determine whether any segmentation parameters need to be adjusted.

In [ ]:
cranberry.detect_fruits(plot = True,
                       plot_size = (10,10))

We now run the morphological analysis with `analyze_morphology()`, which produces an annotated copy of the original image and a DataFrame with the results. In both outputs, each fruit is assigned a unique identifier (`id`) that is useful for cross-referencing the visualizations with the numerical data.

In [ ]:
cranberry.analyze_morphology()

Alternatively, we can examine the tissue segmentation of a specific fruit in detail using `generate_single_fruit_masks()`. The `fruit_id` parameter lets us indicate exactly which fruit from the image or table we want to visualize.

In [ ]:
cranberry.generate_single_fruit_masks(fruit_id = 2)

Finally, we analyze the color of each fruit. By default, `analyze_color()` extracts color information from the `rgb`, `lab`, `hsv`, and `gray` channels for the `total_pericarp`, `outer_pericarp`, `internal_pericarp`, and `locules` tissues.

In this case, we will exclude the locules from the analysis, since being hollow, they only capture the black color of the background. To select specific tissues, we use the `tissue` parameter, and to select specific color channels, we use the `color_space` parameter.

When passing multiple values, they must be written as a comma-separated list, in lowercase, with spaces replaced by `_`. For example:

- RGB and HSV channels: `"rgb, hsv"`
- Locules, total pericarp, and outer pericarp tissues: `"locules, total_pericarp, outer_pericarp"`

If more granular control over which tissues to analyze is needed, the masks obtained with `generate_single_fruit_masks()` allow us to select only the most relevant tissues for the analysis.


In [ ]:
tissues_ext = 'total_pericarp, outer_pericarp, internal_pericarp'

cranberry.analyze_color(tissue = tissues_ext,
                       color_space = 'rgb')

Once `analyze_morphology()` and/or `analyze_color()` have been run, the `results` object becomes available. It holds all analysis outputs along with the methods needed to export them. To save everything in one step, use `save_all()` as shown below.

In [ ]:
cranberry.results.save_all()

We can also export the parameters used during the session with `save_parameters()`, which generates two files: a `.txt` and a `.json`. The `.txt` file is intended for the user and includes the parameters for each method, the Traitly version, the date and time of the analysis, and the image name. The `.json` file, on the other hand, is useful for replicating the analysis — for instance, when processing multiple images with `analyze_folder()` or when running Traitly from the command line.

In [ ]:
cranberry.save_parameters()